# Tutorial 2 — Zadeh's two-doctors paradox

Zadeh (1986) showed that **Dempster's rule of combination can produce dramatically counter-intuitive results** when two sources strongly disagree. This notebook is structured around three questions:

1. *What* does Dempster produce on Zadeh's setup?
2. *Why* does it produce that — what mechanism is at work, and what assumption fails?
3. What do alternative rules (Yager, Dubois-Prade, PCR5/PCR6) do, and how do you choose?

The lesson is methodological: there is no "best" combination rule. Each makes a different commitment about *what to do with conflicting mass*. You pick the rule whose commitment matches your application's risk profile.

**Reference**: Zadeh, L. A. (1986). A simple view of the Dempster-Shafer theory of evidence and its implication for the rule of combination. *AI Magazine*, 7(2), 85–90.

In [ ]:
from carla_evidence import Frame, MassFunction
from carla_evidence.combination import (
    conjunctive,
    dempster,
    dubois_prade,
    mean,
    pcr5,
    pcr6,
    yager,
)

## The setup

A patient is examined by two physicians. The diagnostic frame is

$$\Theta = \{\text{meningitis},\ \text{contusion},\ \text{brain-tumor}\}.$$

Doctor 1 strongly suspects meningitis but allows a 1% chance of brain tumour. Doctor 2 strongly suspects a contusion but also allows 1% for brain tumour.

In [ ]:
theta = Frame.of("meningitis", "contusion", "brain_tumor")

doctor_1 = MassFunction(theta, {("meningitis",): 0.99, ("brain_tumor",): 0.01})
doctor_2 = MassFunction(theta, {("contusion",): 0.99, ("brain_tumor",): 0.01})

print("Doctor 1:", doctor_1)
print("Doctor 2:", doctor_2)

## Step 1: see what's happening *underneath* — the conjunctive product

Dempster's rule and PCR5 both start from the same raw arithmetic — the **conjunctive product** over the cartesian product of focal pairs:

$$m_{\cap}(A) = \sum_{B \cap C = A} m_1(B) \cdot m_2(C).$$

What you call *the conflict* $K$ is just the mass that lands on $\emptyset$. To understand the paradox, look at every term — *which pairs intersect, and where the product mass goes*.

In [ ]:
out_conj = conjunctive(doctor_1, doctor_2)

print("Conjunctive (TBM mode, conflict mass kept on empty-set):")
print(f"  m(empty-set)   = {out_conj.mass(0):.6f}     <-- this is K")
print(f"  m(brain_tumor) = {out_conj.mass(('brain_tumor',)):.6f}")
print(f"  Total          = {sum(m for _, m in out_conj.focals):.6f}")
print()
print("Where each focal pair lands:")
pairs = (
    ("meningitis", "contusion"),
    ("meningitis", "brain_tumor"),
    ("brain_tumor", "contusion"),
    ("brain_tumor", "brain_tumor"),
)
for B, C in pairs:
    p1, p2 = doctor_1.mass((B,)), doctor_2.mass((C,))
    inter = f"{{{B}}}" if B == C else "empty"
    print(f"  d1({{{B:11s}}}) cap d2({{{C:11s}}}) -> {inter:14s} = {p1 * p2:.4f}")

**Three of the four pairs are in conflict.** Their products (`0.9801 + 0.0099 + 0.0099 = 0.9999`) all land on $\emptyset$. Only one pair — `brain_tumor` ∩ `brain_tumor` — survives, with mass `0.0001`.

After this step, three of the four units of evidence have spent themselves disagreeing. **Only `brain_tumor` is left standing on the non-conflicting side.** The rest of the notebook revolves around what to do with that fact.

## Step 2: Dempster — and why it gives the paradox

Dempster's rule takes the conjunctive output, **discards the conflict mass on $\emptyset$**, and **renormalises** the rest by dividing every entry by $1 - K$:

$$m_{1 \oplus 2}(A) = \frac{m_{\cap}(A)}{1 - K} \quad \text{for } A \neq \emptyset.$$

On the Zadeh setup, the only non-conflicting survivor is `brain_tumor` with mass $0.0001$. Renormalising:

$$m_{1 \oplus 2}(\text{brain\_tumor}) = \frac{0.0001}{1 - 0.9999} = \frac{0.0001}{0.0001} = 1.$$

**Certainty.** Out of two doctors who each gave brain-tumour 1%.

In [ ]:
out_dempster = dempster(doctor_1, doctor_2)
print("Dempster:", out_dempster)
print(f"  m(brain_tumor) = {out_dempster.mass(('brain_tumor',))}")

### Why Dempster does this

Dempster's rule encodes a specific philosophical commitment: **"the truth lies in the intersection of the sources' opinions, and the sources are independent and individually reliable."** Under that worldview, conflict is a *statistical artefact* — like two unbiased measurements happening to disagree — that should be normalised away. Whatever survives the conflict, conditional on the joint reliability of the sources, *has to be* the truth.

When the survivor is large (say, two sources mostly agreeing on `meningitis` with mass 0.7 each), this is a perfectly sensible operation. When the survivor is *tiny* (`0.01 · 0.01 = 0.0001`), the same mechanical step amplifies that tiny agreement to certainty.

The paradox isn't a bug in Dempster's rule. It is **a signal that the rule's premises are wrong for this problem**:

- The sources are **not independent** in the strong sense Dempster needs (both doctors are reading the same patient with overlapping symptoms).
- The sources are **not individually reliable** — one of them must be substantially mistaken to reach a diametrically opposite primary diagnosis.
- The frame might be **incomplete** (the true diagnosis might lie outside $\Theta$).

**A high $K$ should be read as informational, not papered over.** Three remediation strategies:

1. Refuse to combine and ask for more evidence (defer / order more tests).
2. Discount the sources before combining (`carla-evidence` Phase 4 — Mercier 2008 contextual discounting).
3. Use a different rule whose conflict policy you trust under high $K$ — Yager, Dubois-Prade, or PCR5/PCR6, below.

`carla-evidence` makes the case extreme by raising `TotalConflictError` when $K = 1$ exactly. The library refuses to silently return `NaN` — see `CLAUDE.md` §"Domain knowledge", trap #2.

## Step 3: alternative rules and their conflict policies

Each alternative rule disagrees with Dempster about *what to do with the conflict mass on $\emptyset$*. None requires the closed-world independence assumption.

### Yager: dump conflict on $\Theta$ ("we don't know")

Yager's rule keeps the conjunctive product, but instead of normalising the conflict away, it sends the conflicting mass to $\Theta$ — admitting *we don't know* rather than artificially boosting one survivor.

In [ ]:
out_yager = yager(doctor_1, doctor_2)
print("Yager:", out_yager)
print(f"m(brain_tumor) = {out_yager.mass(('brain_tumor',)):.6f}")
print(f"m(Theta)       = {out_yager.mass(theta.omega):.6f}")

### Dubois-Prade: redirect conflict to the union

Dubois-Prade redirects each conflicting product `m1(B) · m2(C)` (with `B ∩ C = ∅`) to the **union** `B ∪ C`. The information "the truth is in $B$ or $C$" is preserved, even though we cannot say which.

In [ ]:
out_dp = dubois_prade(doctor_1, doctor_2)
print("Dubois-Prade:", out_dp)
print(f"  m({{meningitis, contusion}}) = {out_dp.mass(('meningitis', 'contusion')):.6f}")
print(f"  m(brain_tumor)               = {out_dp.mass(('brain_tumor',)):.6f}")

Most of the joint mass lands on `{meningitis, contusion}`: "the truth is one of those two; we cannot tell which." This is faithful to what the sources actually said.

### PCR5: redistribute conflict back proportionally

PCR5 takes each conflicting product `m1(B) · m2(C)` and *splits it back* between `B` and `C` proportionally to the sources' commitments — heavier source gets more weight back. The brain-tumour stays at a few permille — *not* certainty.

In [ ]:
out_pcr5 = pcr5(doctor_1, doctor_2)
print("PCR5:", out_pcr5)
print(f"  m(meningitis)  = {out_pcr5.mass(('meningitis',)):.6f}")
print(f"  m(contusion)   = {out_pcr5.mass(('contusion',)):.6f}")
print(f"  m(brain_tumor) = {out_pcr5.mass(('brain_tumor',)):.6f}")

Both leading hypotheses hold roughly half the mass, mirroring the doctors' near-symmetric commitments. The brain-tumour stays in the noise. Decision: defer / order more tests / consult a third opinion.

## Step 4: side-by-side

Read by row: each row is a rule, each column is a hypothesis (or a set), and the entry is the post-fusion mass. The contrast between **Dempster's row** and the others is the entire content of the Zadeh paradox.

In [ ]:
rules = [
    ("Dempster", dempster),
    ("Yager", yager),
    ("Dubois-Prade", dubois_prade),
    ("PCR5", pcr5),
    ("PCR6", pcr6),
    ("Mean", mean),
]

print(
    f"{'rule':<14} {'meningitis':>12} {'contusion':>12} {'brain':>10} {'{m,c}':>10} {'Theta':>10}"
)
for name, rule in rules:
    out = rule(doctor_1, doctor_2)
    print(
        f"{name:<14} "
        f"{out.mass(('meningitis',)):>12.6f} "
        f"{out.mass(('contusion',)):>12.6f} "
        f"{out.mass(('brain_tumor',)):>10.6f} "
        f"{out.mass(('meningitis', 'contusion')):>10.6f} "
        f"{out.mass(theta.omega):>10.6f}"
    )

## Three sources: PCR5 vs PCR6

Per `CLAUDE.md` (§"Domain knowledge — pièges critiques", trap #3), iterating PCR5 over three or more sources is a methodological error — PCR5 is not associative. The library therefore refuses it and points to PCR6 (Martin-Osswald 2006) for multi-source fusion.

In [ ]:
doctor_3 = MassFunction(theta, {("meningitis",): 0.5, ("contusion",): 0.5})

try:
    pcr5([doctor_1, doctor_2, doctor_3])
except NotImplementedError as exc:
    print("PCR5 refused:", exc)

out_pcr6_3src = pcr6([doctor_1, doctor_2, doctor_3])
print("\nPCR6 (3 sources):", out_pcr6_3src)

## Takeaways

1. **The Zadeh paradox is not a Dempster bug. It is a *signal*** that Dempster's premises (closed world, independent reliable sources) are violated for this problem.
2. **Dempster** assumes the truth is the *intersection of the opinions, renormalised*. Use it when conflict is small and you trust the independence-and-reliability premise. Otherwise, beware.
3. **Yager** is conservative: it acknowledges conflict by sending mass to $\Theta$. Useful when you'd rather defer than commit.
4. **Dubois-Prade** keeps the joint support — "the truth is in $B \cup C$". A reasonable middle ground when most conflict is local (`B ∪ C ≠ Theta`).
5. **PCR5/PCR6** redistribute conflict back to the actual sources' hypotheses, weighted by their commitments. Best for decision-making under high conflict.
6. **Mean** is a simple baseline. Idempotent; never amplifies; never sharpens.

Pick deliberately. Document the choice in your fusion pipeline. And never paper over `K = 1` — `carla-evidence` deliberately raises `TotalConflictError` rather than returning `NaN`.